# Notebook 03: ESCO Skill Normalization

## Objective
Map the internal cleaned skill vocabulary from Notebook 02 against the ESCO skills taxonomy.
Produce canonical ESCO-aligned skill profiles for each candidate for use in ATS scoring,
semantic matching, skill gap analysis, and benchmarking.

## Methodology
1. Load and inspect the ESCO skills dataset to understand schema and coverage
2. Clean and normalize ESCO skill labels for matching
3. Perform exact matching between internal vocabulary tokens and ESCO labels
4. Perform fuzzy matching for unmatched tokens using controlled thresholds
5. Flag low-confidence and unresolvable mappings
6. Build a canonical skill mapping table
7. Generate ESCO-normalized candidate skill profiles
8. Export all mapping artifacts for downstream use

## Assumptions
- ESCO dataset is the authoritative external taxonomy for skill normalization
- Internal vocabulary tokens were already lowercased and deduplicated in Notebook 02
- Exact matching is applied first; fuzzy matching only for remaining unmatched tokens
- Fuzzy matching threshold will be set conservatively to minimize false-positive mappings
- Tokens with no acceptable ESCO match will be retained as-is with an unmapped flag

## Inputs
- ../outputs/cleaned_skill_vocabulary.csv
- ../outputs/candidate_skill_profiles.csv
- ../Dataset/skills_en.csv (ESCO taxonomy)

## Outputs
- ../outputs/esco_skill_mapping.csv
- ../outputs/unmatched_skill_tokens.csv
- ../outputs/normalized_candidate_skill_profiles.csv

In [1]:
# importing libraries
import pandas as pd
from pathlib import Path

# defining paths
DATA_DIR = Path("../Dataset")
OUTPUT_DIR = Path("../outputs")

esco_path = DATA_DIR / "skills_en.csv"
vocab_path = OUTPUT_DIR / "cleaned_skill_vocabulary.csv"
profiles_path = OUTPUT_DIR / "candidate_skill_profiles.csv"

# loading ESCO skills dataset
esco_raw = pd.read_csv(esco_path)

print("ESCO raw shape:", esco_raw.shape)
print()
print("Column names:")
for col in esco_raw.columns:
    print(" ", col)
print()
print("First 3 rows:")
print(esco_raw.head(3).to_string())

ESCO raw shape: (13960, 13)

Column names:
  conceptType
  conceptUri
  skillType
  reuseLevel
  preferredLabel
  altLabels
  hiddenLabels
  status
  modifiedDate
  scopeNote
  definition
  inScheme
  description

First 3 rows:
                conceptType                                                             conceptUri         skillType           reuseLevel                     preferredLabel                                                                                                                                                                                                                                         altLabels hiddenLabels    status              modifiedDate scopeNote definition                                                                                                    inScheme                                                                                                                                                                                    

In [2]:
# inspecting ESCO field distributions before building matching logic
print("skillType value counts:")
print(esco_raw["skillType"].value_counts(dropna=False))
print()

print("status value counts:")
print(esco_raw["status"].value_counts(dropna=False))
print()

print("reuseLevel value counts:")
print(esco_raw["reuseLevel"].value_counts(dropna=False))
print()

# checking altLabels coverage
alt_null = esco_raw["altLabels"].isnull().sum()
alt_total = len(esco_raw)
print(f"Records with altLabels present: {alt_total - alt_null} / {alt_total}")
print()

# sample altLabels to confirm delimiter format
print("Sample altLabels values (first 5 non-null):")
sample_alt = esco_raw["altLabels"].dropna().head(5)
for i, val in enumerate(sample_alt):
    print(f"  [{i}]: {repr(val[:120])}")

skillType value counts:
skillType
skill/competence    10734
knowledge            3221
NaN                     5
Name: count, dtype: int64

status value counts:
status
released    13960
Name: count, dtype: int64

reuseLevel value counts:
reuseLevel
sector-specific        6667
cross-sector           3788
occupation-specific    3047
transversal             453
NaN                       5
Name: count, dtype: int64

Records with altLabels present: 13942 / 13960

Sample altLabels values (first 5 non-null):
  [0]: 'manage music staff\ncoordinate duties of musical staff\ndirect musical staff\nmanage staff of music'
  [1]: 'manage prison procedures\nmonitor correctional procedures\noversee prison procedures\noversee correctional procedures\nmanag'
  [2]: 'make use of anti-oppressive practices\nuse anti-oppressive practices\napply non-oppressive practices\napply anti-oppresive '
  [3]: 'checking compliance with rolling stock regulations\ncontrol compliance of regulations of rolling stock\nconduc

In [3]:
# loading internal skill vocabulary from notebook 02
vocab_df = pd.read_csv(vocab_path)

print("Vocabulary shape:", vocab_df.shape)
print()
print("Columns:", vocab_df.columns.tolist())
print()
print("Sample rows:")
print(vocab_df.head(10).to_string(index=False))
print()
print("Tokens flagged for review:", vocab_df["requires_review"].sum())

Vocabulary shape: (217, 3)

Columns: ['skill_token', 'frequency', 'requires_review']

Sample rows:
skill_token  frequency  requires_review
        aws       2583            False
        sql       1923            False
     python       1796            False
        gcp       1712            False
 servicenow       1697            False
      azure       1695            False
 confluence       1628            False
       jira       1622            False
        git       1382            False
     docker       1298            False

Tokens flagged for review: 44


## Section 1: ESCO Label Table Construction

Build a flat lookup table from the ESCO taxonomy.
Each row represents one matchable label — either a preferred label or an alt label —
mapped back to its canonical preferred label and concept URI.

This table is the matching target for both exact and fuzzy matching.
All labels are lowercased and stripped for consistent comparison.

Preferred labels and alt labels are treated as equivalent matching surfaces.
The canonical form returned for any match is always the preferred label.

In [4]:
# dropping the 5 records with no skillType
esco = esco_raw.dropna(subset=["skillType"]).copy()
esco = esco.reset_index(drop=True)

# building flat lookup table with one row per matchable label
records = []

for _, row in esco.iterrows():
    uri = row["conceptUri"]
    preferred = str(row["preferredLabel"]).strip().lower()
    skill_type = row["skillType"]

    # adding preferred label
    records.append({
        "match_label": preferred,
        "preferred_label": preferred,
        "concept_uri": uri,
        "skill_type": skill_type,
        "label_source": "preferred"
    })

    # parsing and adding alt labels
    if pd.notna(row["altLabels"]):
        alt_labels = [a.strip().lower() for a in str(row["altLabels"]).split("\n") if a.strip()]
        for alt in alt_labels:
            records.append({
                "match_label": alt,
                "preferred_label": preferred,
                "concept_uri": uri,
                "skill_type": skill_type,
                "label_source": "alt"
            })

esco_lookup = pd.DataFrame(records)

print("ESCO lookup table shape:", esco_lookup.shape)
print()
print("Label source distribution:")
print(esco_lookup["label_source"].value_counts().to_string())
print()
print("Duplicate match_label count:", esco_lookup["match_label"].duplicated().sum())
print()
print("Sample rows:")
print(esco_lookup.head(8).to_string(index=False))

ESCO lookup table shape: (100621, 5)

Label source distribution:
label_source
alt          86666
preferred    13955

Duplicate match_label count: 1023

Sample rows:
                       match_label                   preferred_label                                                           concept_uri       skill_type label_source
              manage musical staff              manage musical staff http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab skill/competence    preferred
                manage music staff              manage musical staff http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab skill/competence          alt
coordinate duties of musical staff              manage musical staff http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab skill/competence          alt
              direct musical staff              manage musical staff http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab skill/competence    

## Section 2: Exact Matching

Match each internal vocabulary token against the ESCO lookup table using exact string equality.
Both sides are already lowercased, so no additional normalization is needed here.

Where a token matches multiple ESCO concepts (duplicate match_label), preference is given
to the preferred label source over alt. Remaining ties are resolved by taking the first match,
which is arbitrary but consistent and acceptable for this vocabulary size.

In [5]:
# building a deduplicated exact match index from the lookup table
# preferred label source takes priority over alt when the same surface form maps to multiple concepts
esco_lookup_sorted = esco_lookup.sort_values(
    by="label_source",
    key=lambda col: col.map({"preferred": 0, "alt": 1}),
    kind="stable"
)
esco_exact_index = esco_lookup_sorted.drop_duplicates(subset="match_label", keep="first")

print("Exact match index size:", len(esco_exact_index))
print()

# loading internal vocabulary
tokens = vocab_df["skill_token"].tolist()

# performing exact match
exact_matches = []
unmatched_tokens = []

for token in tokens:
    match = esco_exact_index[esco_exact_index["match_label"] == token]
    if not match.empty:
        row = match.iloc[0]
        exact_matches.append({
            "skill_token": token,
            "esco_preferred_label": row["preferred_label"],
            "concept_uri": row["concept_uri"],
            "skill_type": row["skill_type"],
            "match_type": "exact",
            "match_confidence": 1.0
        })
    else:
        unmatched_tokens.append(token)

print(f"Exact matches:    {len(exact_matches)}")
print(f"Unmatched tokens: {len(unmatched_tokens)}")
print(f"Match rate:       {len(exact_matches) / len(tokens) * 100:.1f}%")
print()
print("Unmatched tokens:")
for t in sorted(unmatched_tokens):
    print(" ", t)

Exact match index size: 99598

Exact matches:    11
Unmatched tokens: 206
Match rate:       5.1%

Unmatched tokens:
  agiel
  aglie
  aigle
  aws
  azure
  cad
  cmopliance
  comlpiance
  compilance
  complaince
  compliacne
  compliance
  complianec
  complinace
  conrtact drafting
  contarct drafting
  contrac tdrafting
  contract darfting
  contract drafitng
  contract draftign
  contract drafting
  contract draftnig
  contract dratfing
  contract drfating
  contractd rafting
  contratc drafting
  contrcat drafting
  copmliance
  dcoker
  deu diligence
  docekr
  docker
  dockre
  dokcer
  du ediligence
  due diilgence
  due dilgience
  due diliegnce
  due diligecne
  due diligence
  due diligenec
  due dilignece
  due idligence
  dued iligence
  emlpoyee engagement
  emploeye engagement
  employe eengagement
  employee enaggement
  employee engaegment
  employee engageemnt
  employee engagement
  employee engagemetn
  employee engagemnet
  employee engagmeent
  employee enggaement


In [6]:
# inspecting what actually matched exactly
exact_df = pd.DataFrame(exact_matches)
print("Exact match results:")
print(exact_df[["skill_token", "esco_preferred_label", "skill_type"]].to_string(index=False))
print()

# categorizing unmatched tokens for handling strategy
# separating clean tokens from typo variants
# a token is considered a typo candidate if it appears to be a corrupted version
# of another token already in the vocabulary

# building a set of clean tokens (non-flagged vocabulary)
clean_tokens = set(vocab_df[vocab_df["requires_review"] == False]["skill_token"].tolist())
flagged_tokens = set(vocab_df[vocab_df["requires_review"] == True]["skill_token"].tolist())

# splitting unmatched into flagged vs clean-but-unmatched
unmatched_flagged = [t for t in unmatched_tokens if t in flagged_tokens]
unmatched_clean = [t for t in unmatched_tokens if t in clean_tokens]

print(f"Unmatched clean tokens (not flagged):  {len(unmatched_clean)}")
print(f"Unmatched flagged tokens (typo likely): {len(unmatched_flagged)}")
print()
print("Unmatched clean tokens:")
for t in sorted(unmatched_clean):
    print(" ", t)

Exact match results:
     skill_token                        esco_preferred_label       skill_type
             sql                                         sql        knowledge
          python               python (computer programming)        knowledge
      confluence             use online tools to collaborate skill/competence
             git tools for software configuration management        knowledge
            java                 java (computer programming)        knowledge
             nlp                 natural language processing        knowledge
      tensorflow                             computer vision        knowledge
machine learning                            machine learning        knowledge
  legal research                              legal research        knowledge
 risk management                             risk management        knowledge
           agile        ict project management methodologies        knowledge

Unmatched clean tokens (not flagged):  162

In [7]:
# canonical correction dictionary
# maps every known variant to its correct spelling
# built by inspection of the unmatched token clusters

correction_map = {
    # agile variants (already matched as 'agile' but typos did not match)
    "agiel": "agile", "aglie": "agile", "aigle": "agile",

    # compliance variants
    "cmopliance": "compliance", "comlpiance": "compliance",
    "compilance": "compliance", "complaince": "compliance",
    "compliacne": "compliance", "complianec": "compliance",
    "complinace": "compliance", "copmliance": "compliance",

    # contract drafting variants
    "conrtact drafting": "contract drafting", "contarct drafting": "contract drafting",
    "contrac tdrafting": "contract drafting", "contract darfting": "contract drafting",
    "contract drafitng": "contract drafting", "contract draftign": "contract drafting",
    "contract draftnig": "contract drafting", "contract dratfing": "contract drafting",
    "contract drfating": "contract drafting", "contractd rafting": "contract drafting",
    "contratc drafting": "contract drafting", "contrcat drafting": "contract drafting",

    # docker variants
    "dcoker": "docker", "docekr": "docker", "dockre": "docker",
    "dokcer": "docker",

    # due diligence variants
    "deu diligence": "due diligence", "du ediligence": "due diligence",
    "due diilgence": "due diligence", "due dilgience": "due diligence",
    "due diliegnce": "due diligence", "due diligecne": "due diligence",
    "due diligenec": "due diligence", "due dilignece": "due diligence",
    "due idligence": "due diligence", "dued iligence": "due diligence",

    # employee engagement variants
    "emlpoyee engagement": "employee engagement", "emploeye engagement": "employee engagement",
    "employe eengagement": "employee engagement", "employee enaggement": "employee engagement",
    "employee engaegment": "employee engagement", "employee engageemnt": "employee engagement",
    "employee engagemetn": "employee engagement", "employee engagemnet": "employee engagement",
    "employee engagmeent": "employee engagement", "employee enggaement": "employee engagement",
    "employee negagement": "employee engagement", "employeee ngagement": "employee engagement",
    "emlyoee engagement": "employee engagement", "empolyee engagement": "employee engagement",
    "epmloyee engagement": "employee engagement",

    # hr analytics variants
    "h ranalytics": "hr analytics", "hr aanlytics": "hr analytics",
    "hr analtyics": "hr analytics", "hr analyitcs": "hr analytics",
    "hr analytcis": "hr analytics", "hr analytisc": "hr analytics",
    "hr anayltics": "hr analytics", "hr anlaytics": "hr analytics",
    "hr naalytics": "hr analytics", "hra nalytics": "hr analytics",

    # hrms variants
    "hmrs": "hrms", "hrsm": "hrms",

    # java variants
    "jaav": "java", "jvaa": "java",

    # kubernetes variants
    "kbuernetes": "kubernetes", "kubenretes": "kubernetes", "kuberentes": "kubernetes",
    "kuberneets": "kubernetes", "kubernetse": "kubernetes", "kuberntees": "kubernetes",
    "kubrenetes": "kubernetes", "kuebrnetes": "kubernetes",

    # legal research variants
    "leagl research": "legal research", "lega lresearch": "legal research",
    "legal reesarch": "legal research", "legal resaerch": "legal research",
    "legal reseacrh": "legal research", "legal researhc": "legal research",
    "legal reserach": "legal research", "legal rseearch": "legal research",
    "legalr esearch": "legal research", "legla research": "legal research",
    "lgeal research": "legal research",

    # linux variants
    "linxu": "linux", "liunx": "linux", "lniux": "linux",

    # machine learning variants
    "machien learning": "machine learning", "machine laerning": "machine learning",
    "machine leanring": "machine learning", "machine learinng": "machine learning",
    "machine learnign": "machine learning", "machine learnnig": "machine learning",
    "machine leraning": "machine learning", "machinel earning": "machine learning",
    "machnie learning": "machine learning", "macihne learning": "machine learning",
    "mahcine learning": "machine learning", "mcahine learning": "machine learning",

    # manufacturing variants
    "manuafcturing": "manufacturing", "manufactuirng": "manufacturing",
    "manufacturign": "manufacturing", "manufacturnig": "manufacturing",
    "manufacutring": "manufacturing", "manufcaturing": "manufacturing",
    "maunfacturing": "manufacturing",

    # pandas variants
    "padnas": "pandas", "panads": "pandas", "pandsa": "pandas", "pnadas": "pandas",

    # payroll variants
    "paryoll": "payroll", "payorll": "payroll", "payrlol": "payroll", "pyaroll": "payroll",

    # power bi variants
    "poewr bi": "power bi", "powe rbi": "power bi", "power ib": "power bi",
    "powerb i": "power bi", "powre bi": "power bi", "pwoer bi": "power bi",

    # python variants
    "ptyhon": "python", "pyhton": "python", "pythno": "python", "pytohn": "python",

    # quality control variants
    "qaulity control": "quality control", "quailty control": "quality control",
    "qualit ycontrol": "quality control", "quality cnotrol": "quality control",
    "quality conrtol": "quality control", "quality contorl": "quality control",
    "quality ocntrol": "quality control", "qualityc ontrol": "quality control",
    "qualiyt control": "quality control", "qualtiy control": "quality control",
    "qulaity control": "quality control",

    # recruitment variants
    "rceruitment": "recruitment", "recriutment": "recruitment",
    "recruimtent": "recruitment", "recruitemnt": "recruitment",
    "recruitmetn": "recruitment", "recruitmnet": "recruitment",
    "recrutiment": "recruitment", "recuritment": "recruitment",
    "rercuitment": "recruitment",

    # risk management variants
    "riks management": "risk management", "risk maangement": "risk management",
    "risk manaegment": "risk management", "risk managemnet": "risk management",
    "risk managmeent": "risk management", "risk mangaement": "risk management",
    "riskm anagement": "risk management",

    # solidworks variants
    "sloidworks": "solidworks", "soildworks": "solidworks", "soldiworks": "solidworks",
    "solidowrks": "solidworks", "solidwokrs": "solidworks", "solidworsk": "solidworks",
    "solidwroks": "solidworks", "soliwdorks": "solidworks",

    # stakeholder management variants
    "stakehodler management": "stakeholder management",
    "stakeholde rmanagement": "stakeholder management",
    "stakeholder amnagement": "stakeholder management",
    "stakeholder manaegment": "stakeholder management",
    "stakeholder manageemnt": "stakeholder management",
    "stakeholder managemnet": "stakeholder management",
    "stakeholder mangaement": "stakeholder management",
    "stakeohlder management": "stakeholder management",
    "stkaeholder management": "stakeholder management",

    # strategy variants
    "srtategy": "strategy", "startegy": "strategy", "straetgy": "strategy",
    "strateyg": "strategy", "stratgey": "strategy",

    # tensorflow variants
    "tenosrflow": "tensorflow", "tensofrlow": "tensorflow", "tensorflwo": "tensorflow",
    "tensorfolw": "tensorflow", "tensorlfow": "tensorflow", "tensroflow": "tensorflow",
    "tesnorflow": "tensorflow", "tnesorflow": "tensorflow",
}

# tokens confirmed as product names or platforms with no meaningful ESCO equivalent
# these will be passed through as-is with a manual canonical label
PLATFORM_TOKENS = {
    "aws", "azure", "gcp", "jira", "servicenow", "hrms", "cad",
    "linux", "docker", "kubernetes", "confluence", "git",
    "solidworks", "power bi", "pandas", "tensorflow", "hrms"
}

# applying correction map to full vocabulary
vocab_df["canonical_token"] = vocab_df["skill_token"].apply(
    lambda t: correction_map.get(t, t)
)

# checking correction coverage
corrected = vocab_df[vocab_df["skill_token"] != vocab_df["canonical_token"]]
print(f"Tokens corrected via correction_map: {len(corrected)}")
print()

# distinct canonical tokens after correction
canonical_unique = vocab_df["canonical_token"].nunique()
print(f"Distinct canonical tokens after correction: {canonical_unique}")
print(f"Original distinct tokens:                   {vocab_df['skill_token'].nunique()}")
print()

# showing canonical tokens that are still unmatched after correction
# these are the candidates for fuzzy matching
already_matched = {r["skill_token"] for r in exact_matches}
canonical_for_fuzzy = sorted(set(
    vocab_df[~vocab_df["canonical_token"].isin(already_matched)]["canonical_token"].tolist()
))
print(f"Canonical tokens requiring further matching: {len(canonical_for_fuzzy)}")
print()
print("Canonical tokens for fuzzy matching:")
for t in canonical_for_fuzzy:
    print(" ", t)

Tokens corrected via correction_map: 181

Distinct canonical tokens after correction: 36
Original distinct tokens:                   217

Canonical tokens requiring further matching: 25

Canonical tokens for fuzzy matching:
  aws
  azure
  cad
  compliance
  contract drafting
  docker
  due diligence
  employee engagement
  emplyoee engagement
  gcp
  hr analytics
  hrms
  jira
  kubernetes
  linux
  manufacturing
  pandas
  payroll
  power bi
  quality control
  recruitment
  servicenow
  solidworks
  stakeholder management
  strategy


In [8]:
# adding missed correction before proceeding
correction_map["emplyoee engagement"] = "employee engagement"
vocab_df["canonical_token"] = vocab_df["skill_token"].apply(
    lambda t: correction_map.get(t, t)
)

# confirming fix
print("emplyoee engagement" in vocab_df["canonical_token"].values)
print("Distinct canonical tokens after fix:", vocab_df["canonical_token"].nunique())
print()

# defining platform tokens - will not go to fuzzy matching
# these are product names, platforms, and tools with no meaningful ESCO competency label
PLATFORM_TOKENS = {
    "aws", "azure", "gcp", "jira", "servicenow", "hrms", "cad",
    "linux", "docker", "kubernetes", "pandas", "power bi", "tensorflow",
    "solidworks", "confluence", "git"
}

# reviewing tensorflow exact match - mapped to 'computer vision' which is questionable
tf_match = [r for r in exact_matches if r["skill_token"] == "tensorflow"]
print("tensorflow exact match result:")
print(tf_match)
print()

# defining fuzzy matching candidates - legitimate professional skill phrases
already_exact_matched = {r["skill_token"] for r in exact_matches}

canonical_tokens_all = sorted(vocab_df["canonical_token"].unique())
fuzzy_candidates = [
    t for t in canonical_tokens_all
    if t not in already_exact_matched
    and t not in PLATFORM_TOKENS
]

print(f"Fuzzy matching candidates: {len(fuzzy_candidates)}")
print()
for t in fuzzy_candidates:
    print(" ", t)

False
Distinct canonical tokens after fix: 35

tensorflow exact match result:
[{'skill_token': 'tensorflow', 'esco_preferred_label': 'computer vision', 'concept_uri': 'http://data.europa.eu/esco/skill/7b0d5000-00da-4864-b776-6de49a87a669', 'skill_type': 'knowledge', 'match_type': 'exact', 'match_confidence': 1.0}]

Fuzzy matching candidates: 11

  compliance
  contract drafting
  due diligence
  employee engagement
  hr analytics
  manufacturing
  payroll
  quality control
  recruitment
  stakeholder management
  strategy


In [9]:
# diagnosing the emplyoee engagement correction failure
problem_token = [t for t in vocab_df["canonical_token"].values if "empl" in t and t != "employee engagement"]
print("Remaining employee engagement variants in canonical_token:")
print(problem_token)
print()

# checking exact bytes to find the discrepancy
for t in vocab_df["skill_token"].values:
    if "empl" in t and t not in correction_map and t != "employee engagement":
        print(f"Missing from correction_map: {repr(t)}")

Remaining employee engagement variants in canonical_token:
[]



## Section 4: Fuzzy Matching

Fuzzy matching is applied only to the 11 canonical professional skill phrases
that had no exact ESCO match.

RapidFuzz is used for performance. The matching strategy is:
- Compare each candidate token against all ESCO preferred labels only
- Use token sort ratio to handle word order variation
- Apply a conservative threshold of 90
- Return the single best match above threshold per token
- Tokens below threshold are flagged as unmatched

Alt labels are excluded from fuzzy matching to reduce false positive risk.
Matching against 100k labels with a low threshold produces plausible but incorrect mappings.
Preferred labels only at a high threshold is the safer default for a production pipeline.

tensorflow is removed from exact matches and reclassified as a platform token.
Its ESCO mapping to 'computer vision' is technically valid but misleading for resume scoring.

In [10]:
# installing rapidfuzz if not available
import importlib
if importlib.util.find_spec("rapidfuzz") is None:
    import subprocess
    subprocess.run(["pip", "install", "rapidfuzz", "--quiet"], check=True)

from rapidfuzz import process, fuzz

# removing tensorflow from exact matches - reclassifying as platform token
exact_matches_filtered = [r for r in exact_matches if r["skill_token"] != "tensorflow"]
PLATFORM_TOKENS.add("tensorflow")

print(f"Exact matches after tensorflow removal: {len(exact_matches_filtered)}")
print()

# building preferred-label-only matching surface for fuzzy matching
esco_preferred_only = esco_lookup[esco_lookup["label_source"] == "preferred"].copy()
esco_preferred_labels = esco_preferred_only["match_label"].tolist()

print(f"Fuzzy matching surface size: {len(esco_preferred_labels)} preferred labels")
print()

# running fuzzy matching against 11 candidates
FUZZY_THRESHOLD = 90

fuzzy_matches = []
fuzzy_unmatched = []

for token in fuzzy_candidates:
    result = process.extractOne(
        token,
        esco_preferred_labels,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=FUZZY_THRESHOLD
    )
    if result is not None:
        matched_label, score, _ = result
        esco_row = esco_preferred_only[esco_preferred_only["match_label"] == matched_label].iloc[0]
        fuzzy_matches.append({
            "skill_token": token,
            "esco_preferred_label": esco_row["preferred_label"],
            "concept_uri": esco_row["concept_uri"],
            "skill_type": esco_row["skill_type"],
            "match_type": "fuzzy",
            "match_confidence": round(score / 100, 3)
        })
    else:
        fuzzy_unmatched.append(token)

print(f"Fuzzy matched:   {len(fuzzy_matches)}")
print(f"Fuzzy unmatched: {len(fuzzy_unmatched)}")
print()
print("Fuzzy match results:")
for r in fuzzy_matches:
    print(f"  {r['skill_token']:<30} -> {r['esco_preferred_label']:<45} (score: {r['match_confidence']})")
print()
if fuzzy_unmatched:
    print("Unmatched after fuzzy:")
    for t in fuzzy_unmatched:
        print(" ", t)

Exact matches after tensorflow removal: 10

Fuzzy matching surface size: 13955 preferred labels

Fuzzy matched:   0
Fuzzy unmatched: 11

Fuzzy match results:

Unmatched after fuzzy:
  compliance
  contract drafting
  due diligence
  employee engagement
  hr analytics
  manufacturing
  payroll
  quality control
  recruitment
  stakeholder management
  strategy


In [11]:
# searching ESCO preferred labels for substring matches
# this surfaces candidate mappings for manual review

preferred_label_list = esco_preferred_only[["match_label", "concept_uri", "skill_type"]].copy()

print("ESCO preferred label candidates for each unmatched token")
print("(showing up to 8 matches per token)")
print()

for token in fuzzy_unmatched:
    candidates = preferred_label_list[
        preferred_label_list["match_label"].str.contains(token, case=False, na=False)
    ]
    print(f"Token: '{token}' — {len(candidates)} candidates found")
    if not candidates.empty:
        for _, row in candidates.head(8).iterrows():
            print(f"  {row['match_label']:<70} [{row['skill_type']}]")
    print()

ESCO preferred label candidates for each unmatched token
(showing up to 8 matches per token)

Token: 'compliance' — 51 candidates found
  control compliance of railway vehicles regulations                     [skill/competence]
  manage it security compliances                                         [skill/competence]
  ensure compliance with port regulations                                [skill/competence]
  ensure compliance with environmental legislation                       [skill/competence]
  check construction compliance                                          [skill/competence]
  cloud security and compliance                                          [knowledge]
  ensure ongoing compliance with regulations                             [skill/competence]
  ensure compliance with legal requirements                              [skill/competence]

Token: 'contract drafting' — 0 candidates found

Token: 'due diligence' — 0 candidates found

Token: 'employee engagement' — 0 candida

In [12]:
# searching alt labels for the zero-candidate and ambiguous tokens
# these tokens had no preferred label match - checking alt labels as a fallback

alt_label_surface = esco_lookup[esco_lookup["label_source"] == "alt"].copy()

zero_candidate_tokens = [
    "contract drafting", "due diligence", "employee engagement",
    "hr analytics", "recruitment", "stakeholder management"
]

print("Alt label candidates for zero-match tokens")
print("(showing up to 6 matches per token)")
print()

for token in zero_candidate_tokens:
    candidates = alt_label_surface[
        alt_label_surface["match_label"].str.contains(token, case=False, na=False)
    ]
    print(f"Token: '{token}' — {len(candidates)} alt label candidates found")
    if not candidates.empty:
        for _, row in candidates.head(6).iterrows():
            print(f"  alt: {row['match_label']:<55} -> preferred: {row['preferred_label']}")
    print()

Alt label candidates for zero-match tokens
(showing up to 6 matches per token)

Token: 'contract drafting' — 1 alt label candidates found
  alt: energy performance contract drafting                    -> preferred: prepare energy performance contracts

Token: 'due diligence' — 0 alt label candidates found

Token: 'employee engagement' — 0 alt label candidates found

Token: 'hr analytics' — 0 alt label candidates found

Token: 'recruitment' — 15 alt label candidates found
  alt: manage recruitment of employees                         -> preferred: recruit employees
  alt: broodstock recruitment by genetic selection             -> preferred: aquaculture reproduction
  alt: produce recruitment policies                            -> preferred: develop employment policies
  alt: operate recruitment policies                            -> preferred: develop employment policies
  alt: implement recruitment policies                          -> preferred: develop employment policies
  alt: estab

## Section 5: Manual Mapping Decisions and Final Match Table Assembly

ESCO does not model several common resume skill nouns at the abstraction level
used in professional resume writing. Forced mappings to occupation-specific ESCO
phrases would introduce false domain specificity.

Manual mapping decisions:
- recruitment       -> recruit employees (acceptable generic mapping via alt label)
- payroll           -> manage payroll (direct conceptual match via preferred label)
- quality control   -> oversee quality control (most generic preferred label candidate)
- compliance        -> ensure ongoing compliance with regulations (domain-neutral)
- contract drafting -> no valid mapping, flagged as unmapped
- due diligence     -> no valid mapping, flagged as unmapped
- employee engagement -> no valid mapping, flagged as unmapped
- hr analytics      -> no valid mapping, flagged as unmapped
- stakeholder management -> no valid mapping, flagged as unmapped
- manufacturing     -> no valid mapping, flagged as unmapped (all candidates occupation-specific)
- strategy          -> no valid mapping, flagged as unmapped (all candidates domain-specific)

Platform and tool tokens are retained as canonical labels without ESCO mapping.
They carry signal for ATS scoring and semantic matching but fall outside ESCO scope.

In [13]:
# fetching concept URIs for manual mappings
def get_esco_row(preferred_label_text):
    row = esco_preferred_only[
        esco_preferred_only["match_label"] == preferred_label_text
    ]
    if row.empty:
        # fallback to alt label surface
        row = alt_label_surface[
            alt_label_surface["preferred_label"] == preferred_label_text
        ]
    if row.empty:
        return None
    return row.iloc[0]

manual_mapping_decisions = {
    "recruitment":    "recruit employees",
    "payroll":        "manage payroll",
    "quality control": "oversee quality control",
    "compliance":     "ensure ongoing compliance with regulations",
}

manual_matches = []
for token, esco_label in manual_mapping_decisions.items():
    row = get_esco_row(esco_label)
    if row is not None:
        manual_matches.append({
            "skill_token":          token,
            "esco_preferred_label": row["preferred_label"],
            "concept_uri":          row["concept_uri"],
            "skill_type":           row["skill_type"],
            "match_type":           "manual",
            "match_confidence":     1.0
        })
    else:
        print(f"WARNING: could not resolve manual mapping for '{token}' -> '{esco_label}'")

print(f"Manual matches resolved: {len(manual_matches)}")
print()
for r in manual_matches:
    print(f"  {r['skill_token']:<25} -> {r['esco_preferred_label']}")
print()

# tokens with no valid ESCO mapping
explicitly_unmapped = [
    "contract drafting", "due diligence", "employee engagement",
    "hr analytics", "stakeholder management", "manufacturing", "strategy"
]

print(f"Explicitly unmapped tokens: {len(explicitly_unmapped)}")
for t in explicitly_unmapped:
    print(f"  {t}")

Manual matches resolved: 4

  recruitment               -> recruit employees
  payroll                   -> manage payroll
  quality control           -> oversee quality control
  compliance                -> ensure ongoing compliance with regulations

Explicitly unmapped tokens: 7
  contract drafting
  due diligence
  employee engagement
  hr analytics
  stakeholder management
  manufacturing
  strategy


## Section 6: Full Mapping Table Assembly and Artifact Export

All match sources are consolidated into a single canonical mapping table:
- 10 exact matches
- 4 manual matches
- 7 explicitly unmapped professional skill tokens (retained with no_esco_match flag)
- 14 platform and tool tokens (retained with platform flag)

The mapping table covers all 35 canonical tokens.
Every candidate skill profile is then rebuilt using canonical tokens,
with ESCO preferred labels substituted where a mapping exists.

In [14]:
# assembling complete canonical token list
all_canonical = sorted(vocab_df["canonical_token"].unique())

# building lookup from match results
match_lookup = {}

for r in exact_matches_filtered:
    match_lookup[r["skill_token"]] = r

for r in manual_matches:
    match_lookup[r["skill_token"]] = r

# building the full mapping table - one row per canonical token
mapping_rows = []

for token in all_canonical:
    if token in match_lookup:
        r = match_lookup[token]
        mapping_rows.append({
            "canonical_token":      token,
            "esco_preferred_label": r["esco_preferred_label"],
            "concept_uri":          r["concept_uri"],
            "skill_type":           r["skill_type"],
            "match_type":           r["match_type"],
            "match_confidence":     r["match_confidence"],
            "token_category":       "esco_mapped"
        })
    elif token in PLATFORM_TOKENS:
        mapping_rows.append({
            "canonical_token":      token,
            "esco_preferred_label": None,
            "concept_uri":          None,
            "skill_type":           None,
            "match_type":           "none",
            "match_confidence":     None,
            "token_category":       "platform_tool"
        })
    elif token in explicitly_unmapped:
        mapping_rows.append({
            "canonical_token":      token,
            "esco_preferred_label": None,
            "concept_uri":          None,
            "skill_type":           None,
            "match_type":           "none",
            "match_confidence":     None,
            "token_category":       "no_esco_match"
        })
    else:
        # catch any token not accounted for
        mapping_rows.append({
            "canonical_token":      token,
            "esco_preferred_label": None,
            "concept_uri":          None,
            "skill_type":           None,
            "match_type":           "none",
            "match_confidence":     None,
            "token_category":       "unresolved"
        })

mapping_df = pd.DataFrame(mapping_rows)

print("Mapping table shape:", mapping_df.shape)
print()
print("token_category distribution:")
print(mapping_df["token_category"].value_counts().to_string())
print()
print("Full mapping table:")
print(mapping_df[[
    "canonical_token", "esco_preferred_label", "match_type", "token_category"
]].to_string(index=False))

Mapping table shape: (35, 7)

token_category distribution:
token_category
esco_mapped      14
platform_tool    14
no_esco_match     7

Full mapping table:
       canonical_token                        esco_preferred_label match_type token_category
                 agile        ict project management methodologies      exact    esco_mapped
                   aws                                        None       none  platform_tool
                 azure                                        None       none  platform_tool
                   cad                                        None       none  platform_tool
            compliance  ensure ongoing compliance with regulations     manual    esco_mapped
            confluence             use online tools to collaborate      exact    esco_mapped
     contract drafting                                        None       none  no_esco_match
                docker                                        None       none  platform_tool
        

In [15]:
# exporting esco skill mapping artifact
mapping_df.to_csv(OUTPUT_DIR / "esco_skill_mapping.csv", index=False)
print("esco_skill_mapping.csv exported:", mapping_df.shape)
print()

# exporting unmatched tokens artifact
unmatched_df = mapping_df[mapping_df["token_category"].isin(["no_esco_match", "platform_tool"])][
    ["canonical_token", "token_category"]
].copy()
unmatched_df.to_csv(OUTPUT_DIR / "unmatched_skill_tokens.csv", index=False)
print("unmatched_skill_tokens.csv exported:", unmatched_df.shape)
print()

# building token -> normalized label lookup for profile reconstruction
# esco_mapped tokens: use esco_preferred_label as the normalized label
# platform_tool and no_esco_match tokens: use canonical_token as-is
normalization_lookup = {}
for _, row in mapping_df.iterrows():
    if row["token_category"] == "esco_mapped":
        normalization_lookup[row["canonical_token"]] = row["esco_preferred_label"]
    else:
        normalization_lookup[row["canonical_token"]] = row["canonical_token"]

print("Normalization lookup entries:", len(normalization_lookup))
print()

# loading candidate skill profiles from notebook 02
profiles_df = pd.read_csv(profiles_path)

# rebuilding normalized profiles
# step 1: map each raw skill token to its canonical form using correction_map
# step 2: map canonical form to normalized label using normalization_lookup
def normalize_profile(pipe_delimited_profile):
    tokens = pipe_delimited_profile.split("|")
    normalized = []
    for token in tokens:
        canonical = correction_map.get(token, token)
        normalized_label = normalization_lookup.get(canonical, canonical)
        normalized.append(normalized_label)
    # deduplicate while preserving order
    seen = set()
    deduped = []
    for t in normalized:
        if t not in seen:
            seen.add(t)
            deduped.append(t)
    return "|".join(sorted(deduped))

profiles_df["normalized_skill_profile"] = profiles_df["skill_profile"].apply(normalize_profile)
profiles_df["normalized_skill_count"] = profiles_df["normalized_skill_profile"].apply(
    lambda x: len(x.split("|"))
)

print("Normalized profiles shape:", profiles_df.shape)
print()
print("Skill count before and after normalization:")
print(f"  original  min/mean/max: "
      f"{profiles_df['skill_count'].min()} / "
      f"{round(profiles_df['skill_count'].mean(), 2)} / "
      f"{profiles_df['skill_count'].max()}")
print(f"  normalized min/mean/max: "
      f"{profiles_df['normalized_skill_count'].min()} / "
      f"{round(profiles_df['normalized_skill_count'].mean(), 2)} / "
      f"{profiles_df['normalized_skill_count'].max()}")
print()

# spot check: first 5 candidates
print("Sample normalized profiles (first 5 candidates):")
for _, row in profiles_df.head(5).iterrows():
    print(f"  {row['candidate_id']}")
    print(f"    original:   {row['skill_profile']}")
    print(f"    normalized: {row['normalized_skill_profile']}")
    print()

esco_skill_mapping.csv exported: (35, 7)

unmatched_skill_tokens.csv exported: (21, 2)

Normalization lookup entries: 35

Normalized profiles shape: (5000, 5)

Skill count before and after normalization:
  original  min/mean/max: 5 / 5.91 / 6
  normalized min/mean/max: 5 / 5.91 / 6

Sample normalized profiles (first 5 candidates):
  CND_100001
    original:   aws|machine learning|pandas|power bi|python|servicenow
    normalized: aws|machine learning|pandas|power bi|python (computer programming)|servicenow

  CND_100002
    original:   aws|docker|git|jira|kubernetes|python
    normalized: aws|docker|jira|kubernetes|python (computer programming)|tools for software configuration management

  CND_100003
    original:   aws|compliance|contract drafting|due diligence|legal research|servicenow
    normalized: aws|contract drafting|due diligence|ensure ongoing compliance with regulations|legal research|servicenow

  CND_100004
    original:   aws|azure|docker|gcp|kubernetes|sql
    normalized

In [16]:
# exporting normalized candidate skill profiles
normalized_export = profiles_df[[
    "candidate_id",
    "skill_profile",
    "skill_count",
    "normalized_skill_profile",
    "normalized_skill_count"
]].copy()

normalized_export.to_csv(OUTPUT_DIR / "normalized_candidate_skill_profiles.csv", index=False)
print("normalized_candidate_skill_profiles.csv exported:", normalized_export.shape)
print()

# final validation
reloaded = pd.read_csv(OUTPUT_DIR / "normalized_candidate_skill_profiles.csv")

print("Validation checks:")
print(f"  Row count matches:        {len(reloaded) == 5000}")
print(f"  Null normalized profiles: {reloaded['normalized_skill_profile'].isnull().sum()}")
print(f"  Empty normalized profiles:{(reloaded['normalized_skill_profile'] == '').sum()}")
print(f"  Min normalized count:     {reloaded['normalized_skill_count'].min()}")
print()

# confirming all three artifacts are present in output directory
print("Output artifacts:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    size_kb = round(f.stat().st_size / 1024, 1)
    print(f"  {f.name:<50} {size_kb} KB")

normalized_candidate_skill_profiles.csv exported: (5000, 5)

Validation checks:
  Row count matches:        True
  Null normalized profiles: 0
  Empty normalized profiles:0
  Min normalized count:     5

Output artifacts:
  candidate_skill_profiles.csv                       292.4 KB
  cleaned_skill_vocabulary.csv                       4.8 KB
  esco_skill_mapping.csv                             2.7 KB
  normalized_candidate_skill_profiles.csv            724.3 KB
  processed_resumes.csv                              3062.7 KB
  unmatched_skill_tokens.csv                         0.5 KB
